In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [ ]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['mmorise/World']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

In [ ]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

In [ ]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [ ]:
# check if it has the right content
dfinf.head(1)

In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='your_net_id_here'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

## WoC Project Statistics

### annkennedy_tracer
- Number of commits: 366
- Number of authors: 4
- Max time: 1624573166
- Min time: 1499548710

### brentp_vcfanno
- Number of commits: 428
- Number of authors: 17
- Max time: 1756585014
- Min time: 1429908668

### freesurfer_freesurfer
- Number of commits: 41070
- Number of authors: 238
- Max time: 1762559181
- Min time: 825262903

### isa-tools_isa-api
- Number of commits: 4134
- Number of authors: 52
- Max time: 1774857269
- Min time: 1426161039

### markmfredrickson_ritools
- Number of commits: 1118
- Number of authors: 20
- Max time: 1761340147
- Min time: 1307049689

### neuroelectro_neuroelectro_org
- Number of commits: 1259
- Number of authors: 38
- Max time: 1741605788
- Min time: 1357677337

### nvidiagameworks_falcor
- Number of commits: 1954
- Number of authors: 90
- Max time: 1757340318
- Min time: 1501098582

### openforcefield_openforcefield
- Number of commits: 9643
- Number of authors: 120
- Max time: 1762466664
- Min time: 1465668694

### r-spatial_mapedit
- Number of commits: 441
- Number of authors: 20
- Max time: 1745214706
- Min time: 1484853293

### ropensci_textreuse
- Number of commits: 309
- Number of authors: 17
- Max time: 1741080545
- Min time: 1426363160


## Part 1 Project Statistics

### neuroelectro_neuroelectro_org
GitHub:
- Stars: 19
- Forks: 7
- Last commit date: 2024-12-07T16:21:16Z

World of Code:
- Number of commits: 1259
- Number of authors: 38
- Max time: 1741605788
- Min time: 1357677337

### ropensci_textreuse
GitHub:
- Stars: 200
- Forks: 34
- Last commit date: 2026-07-25T01:12:17Z

World of Code:
- Number of commits: 309
- Number of authors: 17
- Max time: 1741080545
- Min time: 1426363160

### freesurfer_freesurfer
GitHub:
- Stars: 877
- Forks: 301
- Last commit date: 2026-09-29T16:24:42Z

World of Code:
- Number of commits: 41070
- Number of authors: 238
- Max time: 1762559181
- Min time: 825262903

### openforcefield_openforcefield
GitHub:
- Stars: 415
- Forks: 107
- Last commit date: 2026-09-28T19:46:56Z

World of Code:
- Number of commits: 9643
- Number of authors: 120
- Max time: 1762466664
- Min time: 1465668694

### markmfredrickson_ritools
GitHub:
- Stars: 17
- Forks: 11
- Last commit date: 2025-05-17T13:18:53Z

World of Code:
- Number of commits: 1118
- Number of authors: 20
- Max time: 1761340147
- Min time: 1307049689

### annkennedy_tracer
GitHub:
- Stars: 57
- Forks: 9
- Last commit date: 2025-01-22T06:50:08Z

World of Code:
- Number of commits: 366
- Number of authors: 4
- Max time: 1624573166
- Min time: 1499548710

### brentp_vcfanno
GitHub:
- Stars: 403
- Forks: 56
- Last commit date: 2026-06-16T19:02:01Z

World of Code:
- Number of commits: 428
- Number of authors: 17
- Max time: 1756585014
- Min time: 1429908668

### r-spatial_mapedit
GitHub:
- Stars: 221
- Forks: 33
- Last commit date: 2026-02-15T11:31:59Z

World of Code:
- Number of commits: 441
- Number of authors: 20
- Max time: 1745214706
- Min time: 1484853293

### isa-tools_isa-api
GitHub:
- Stars: 48
- Forks: 42
- Last commit date: 2026-07-23T08:16:52Z

World of Code:
- Number of commits: 4134
- Number of authors: 52
- Max time: 1774857269
- Min time: 1426161039

### nvidiagameworks_falcor
GitHub:
- Stars: 3242
- Forks: 622
- Last commit date: 2026-09-21T15:08:34Z

World of Code:
- Number of commits: 1954
- Number of authors: 90
- Max time: 1757340318
- Min time: 1501098582
